# Imports

In [561]:
%pip install -q requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement requirements.txt (from versions: none)
ERROR: No matching distribution found for requirements.txt


In [562]:
import yfinance as yf
import pandas as pd
import numpy as np

# Constants

## Control

In [563]:
FETCH_DATA = False

## Routes

In [564]:
BITCOIN_DATA = 'bitcoin.csv'

## Dataset

In [565]:
TICKER = "BTC-USD"
START_DATE = "2024-10-01"
END_DATE = "2025-10-01"

## Indicator

In [566]:
THETA_ADX = 25
ADX_LOOKBACK = 14

## Strategy

In [567]:
SMA_SHORT = 20
SMA_LONG = 50
RSI_LOOKBACK = 14

# Functions

## Fetching data

In [568]:
def fetch_from_yfinance(ticker: str, route: str, start_date, end_date, fetch: bool=True):
    if fetch:
        df = yf.download(
            tickers=ticker,
            start=start_date,
            end=end_date
        )
        df.to_csv(route)
        df = pd.read_csv(route, skiprows=[1, 2], header=0)
        df = df.rename(columns={'Price': 'Date'})
        df = df.set_index('Date')
        df.to_csv(route)
    else:
        df = pd.read_csv(route)
    return df

# Fetch data

In [569]:
df = fetch_from_yfinance(TICKER, BITCOIN_DATA, START_DATE, END_DATE, FETCH_DATA)
df

,Date,Close,High,Low,Open,Volume,Return
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362
...,...,...,...,...,...,...,...
360,2025-09-26,109712.828125,110359.195312,108728.976562,109041.296875,57738288949,0.006085
361,2025-09-27,109681.945312,109778.500000,109144.296875,109707.140625,26308042910,-0.000281
362,2025-09-28,112122.640625,112375.484375,109236.945312,109681.945312,33371048505,0.022252
363,2025-09-29,114400.382812,114473.570312,111589.953125,112117.875000,60000147466,0.020315


# Preprocessing

# Indicators

## ADX

### Directional Moving (DM)

$$ +DM_t = H_{t} - H_{t-1} $$
$$ -DM_t = L_{t-1} - L_{t} $$

In [570]:
df['DMP'] = df['High'].diff()
df['DMN'] = df['Low'].diff()

def apply_dm_logic(row):
    dmp = row['DMP']
    dmn = row['DMN']
    
    dm_plus = dmp if dmp > 0 else 0
    dm_minus = -dmn if dmn < 0 else 0 
    
    if dm_plus > 0 and dm_minus > 0:
        if dm_plus > dm_minus:
            dm_minus = 0
        else:
            dm_plus = 0
            
    return pd.Series({'DM_Plus': dm_plus, 'DM_Minus': dm_minus})

dm_results = df.apply(apply_dm_logic, axis=1)

df['+DM'] = dm_results['DM_Plus']
df['-DM'] = dm_results['DM_Minus']

df = df.drop(columns=['DMP', 'DMN'])
df.head()

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN,0.000000,0.000000
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357,0.000000,192.328125
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088,0.000000,118.144531
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529,996.953125,0.000000
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362,0.000000,0.000000


### True Range (TR)

$$TR_{t} = max(H_{t}-L_{t}, |H_{t}-C_{t-1}|, |L_{t}-C_{t-1}|)$$

In [571]:
df['High_Low'] = df['High'] - df['Low']
df['High_PrevClose'] = abs(df['High'] - df['Close'].shift(1))
df['Low_PrevClose'] = abs(df['Low'] - df['Close'].shift(1))
df['TR'] = df[['High_Low', 'High_PrevClose', 'Low_PrevClose']].max(axis=1)

df = df.drop(columns=['High_Low', 'High_PrevClose', 'Low_PrevClose'])
df.head()

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN,0.000000,0.000000,3921.703125
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357,0.000000,192.328125,2360.738281
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088,0.000000,118.144531,1590.234375
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529,996.953125,0.000000,2006.050781
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362,0.000000,0.000000,681.441406


### Average True Range (ATR)

$$ATR_{t} = \frac{1}{n} \sum_{i=t-n+1}^{t} TR_{i}$$

In [572]:
df['ATR'] = df['TR'].rolling(window=ADX_LOOKBACK).mean()

df.head(ADX_LOOKBACK*2).tail(ADX_LOOKBACK)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR
14,2024-10-15,67041.109375,67881.679688,64809.195312,66050.367188,48863870879,0.015065,1399.187500,0.000000,3072.484375,2045.187500
15,2024-10-16,67612.718750,68375.289062,66758.726562,67042.460938,38195189534,0.008526,493.609375,0.000000,1616.562500,1992.032087
16,2024-10-17,67399.835938,67912.210938,66647.390625,67617.078125,32790898511,-0.003149,0.000000,111.335938,1264.820312,1968.788225
17,2024-10-18,68418.789062,68969.750000,67177.820312,67419.109375,36857165014,0.015118,1057.539062,0.000000,1791.929688,1953.493862
18,2024-10-19,68362.734375,68668.007812,68024.640625,68418.976562,14443497908,-0.000819,0.000000,0.000000,643.367188,1950.774275
19,2024-10-20,69001.703125,69359.007812,68105.718750,68364.179688,18975847518,0.009347,691.000000,0.000000,1253.289062,1959.836217
20,2024-10-21,67367.851562,69462.734375,66829.851562,69002.000000,37498611780,-0.023678,0.000000,1275.867188,2632.882812,1984.245257
21,2024-10-22,67361.406250,67801.578125,66581.367188,67360.703125,31808472566,-0.000096,0.000000,248.484375,1220.210938,1976.350167
22,2024-10-23,66432.195312,67402.742188,65188.035156,67362.375000,32263980353,-0.013794,0.000000,1393.332031,2214.707031,1977.813337
23,2024-10-24,68161.054688,68798.960938,66454.101562,66653.703125,31414428647,0.026024,1396.218750,0.000000,2366.765625,1979.616908


### Simple Moving Average Directional Movement (SMA DM)

$$SMA+DM = \frac{1}{n} \sum_{i=t-n+1}^{t}+DM_{i}$$
$$SMA-DM = \frac{1}{n} \sum_{i=t-n+1}^{t}-DM_{i}$$

In [573]:
df['SMA+DM'] = df['+DM'].rolling(window=ADX_LOOKBACK).mean()
df['SMA-DM'] = df['-DM'].rolling(window=ADX_LOOKBACK).mean()

df.head(ADX_LOOKBACK*2).tail(ADX_LOOKBACK)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR,SMA+DM,SMA-DM
14,2024-10-15,67041.109375,67881.679688,64809.195312,66050.367188,48863870879,0.015065,1399.187500,0.000000,3072.484375,2045.187500,706.480469,283.960658
15,2024-10-16,67612.718750,68375.289062,66758.726562,67042.460938,38195189534,0.008526,493.609375,0.000000,1616.562500,1992.032087,741.738281,270.222935
16,2024-10-17,67399.835938,67912.210938,66647.390625,67617.078125,32790898511,-0.003149,0.000000,111.335938,1264.820312,1968.788225,741.738281,269.736607
17,2024-10-18,68418.789062,68969.750000,67177.820312,67419.109375,36857165014,0.015118,1057.539062,0.000000,1791.929688,1953.493862,746.065848,269.736607
18,2024-10-19,68362.734375,68668.007812,68024.640625,68418.976562,14443497908,-0.000819,0.000000,0.000000,643.367188,1950.774275,746.065848,269.736607
19,2024-10-20,69001.703125,69359.007812,68105.718750,68364.179688,18975847518,0.009347,691.000000,0.000000,1253.289062,1959.836217,753.383929,269.736607
20,2024-10-21,67367.851562,69462.734375,66829.851562,69002.000000,37498611780,-0.023678,0.000000,1275.867188,2632.882812,1984.245257,647.374163,360.869978
21,2024-10-22,67361.406250,67801.578125,66581.367188,67360.703125,31808472566,-0.000096,0.000000,248.484375,1220.210938,1976.350167,647.374163,356.548270
22,2024-10-23,66432.195312,67402.742188,65188.035156,67362.375000,32263980353,-0.013794,0.000000,1393.332031,2214.707031,1977.813337,647.374163,346.861328
23,2024-10-24,68161.054688,68798.960938,66454.101562,66653.703125,31414428647,0.026024,1396.218750,0.000000,2366.765625,1979.616908,747.104074,245.475167


### Directional Indicator (DI)

$$+DI_{t} = (\frac{SMA+DM_{t}}{ATR_{t}}) \times 100$$
$$-DI_{t} = (\frac{SMA-DM_{t}}{ATR_{t}}) \times 100$$

In [574]:
df['+DI'] = df['SMA+DM'] / df['ATR'] * 100
df['-DI'] = df['SMA-DM'] / df['ATR'] * 100

df.head(ADX_LOOKBACK*2).tail(ADX_LOOKBACK)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR,SMA+DM,SMA-DM,+DI,-DI
14,2024-10-15,67041.109375,67881.679688,64809.195312,66050.367188,48863870879,0.015065,1399.187500,0.000000,3072.484375,2045.187500,706.480469,283.960658,34.543555,13.884334
15,2024-10-16,67612.718750,68375.289062,66758.726562,67042.460938,38195189534,0.008526,493.609375,0.000000,1616.562500,1992.032087,741.738281,270.222935,37.235258,13.565190
16,2024-10-17,67399.835938,67912.210938,66647.390625,67617.078125,32790898511,-0.003149,0.000000,111.335938,1264.820312,1968.788225,741.738281,269.736607,37.674864,13.700641
17,2024-10-18,68418.789062,68969.750000,67177.820312,67419.109375,36857165014,0.015118,1057.539062,0.000000,1791.929688,1953.493862,746.065848,269.736607,38.191359,13.807907
18,2024-10-19,68362.734375,68668.007812,68024.640625,68418.976562,14443497908,-0.000819,0.000000,0.000000,643.367188,1950.774275,746.065848,269.736607,38.244602,13.827156
19,2024-10-20,69001.703125,69359.007812,68105.718750,68364.179688,18975847518,0.009347,691.000000,0.000000,1253.289062,1959.836217,753.383929,269.736607,38.441168,13.763222
20,2024-10-21,67367.851562,69462.734375,66829.851562,69002.000000,37498611780,-0.023678,0.000000,1275.867188,2632.882812,1984.245257,647.374163,360.869978,32.625713,18.186763
21,2024-10-22,67361.406250,67801.578125,66581.367188,67360.703125,31808472566,-0.000096,0.000000,248.484375,1220.210938,1976.350167,647.374163,356.548270,32.756046,18.040744
22,2024-10-23,66432.195312,67402.742188,65188.035156,67362.375000,32263980353,-0.013794,0.000000,1393.332031,2214.707031,1977.813337,647.374163,346.861328,32.731813,17.537617
23,2024-10-24,68161.054688,68798.960938,66454.101562,66653.703125,31414428647,0.026024,1396.218750,0.000000,2366.765625,1979.616908,747.104074,245.475167,37.739831,12.400135


### Directional Index (DX)

$$DX_{t} = \frac{|(+DI_{t})-(-DI_{t})|}{|(+DI_{t})+(-DI_{t})|} \times 100$$

In [575]:
df['DX'] = abs(df['+DI'] - df['-DI'])/abs(df['+DI'] + df['-DI']) * 100

df.head(ADX_LOOKBACK*2).tail(ADX_LOOKBACK)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR,SMA+DM,SMA-DM,+DI,-DI,DX
14,2024-10-15,67041.109375,67881.679688,64809.195312,66050.367188,48863870879,0.015065,1399.187500,0.000000,3072.484375,2045.187500,706.480469,283.960658,34.543555,13.884334,42.659760
15,2024-10-16,67612.718750,68375.289062,66758.726562,67042.460938,38195189534,0.008526,493.609375,0.000000,1616.562500,1992.032087,741.738281,270.222935,37.235258,13.565190,46.594211
16,2024-10-17,67399.835938,67912.210938,66647.390625,67617.078125,32790898511,-0.003149,0.000000,111.335938,1264.820312,1968.788225,741.738281,269.736607,37.674864,13.700641,46.664695
17,2024-10-18,68418.789062,68969.750000,67177.820312,67419.109375,36857165014,0.015118,1057.539062,0.000000,1791.929688,1953.493862,746.065848,269.736607,38.191359,13.807907,46.891917
18,2024-10-19,68362.734375,68668.007812,68024.640625,68418.976562,14443497908,-0.000819,0.000000,0.000000,643.367188,1950.774275,746.065848,269.736607,38.244602,13.827156,46.891917
19,2024-10-20,69001.703125,69359.007812,68105.718750,68364.179688,18975847518,0.009347,691.000000,0.000000,1253.289062,1959.836217,753.383929,269.736607,38.441168,13.763222,47.271783
20,2024-10-21,67367.851562,69462.734375,66829.851562,69002.000000,37498611780,-0.023678,0.000000,1275.867188,2632.882812,1984.245257,647.374163,360.869978,32.625713,18.186763,28.416152
21,2024-10-22,67361.406250,67801.578125,66581.367188,67360.703125,31808472566,-0.000096,0.000000,248.484375,1220.210938,1976.350167,647.374163,356.548270,32.756046,18.040744,28.968960
22,2024-10-23,66432.195312,67402.742188,65188.035156,67362.375000,32263980353,-0.013794,0.000000,1393.332031,2214.707031,1977.813337,647.374163,346.861328,32.731813,17.537617,30.225519
23,2024-10-24,68161.054688,68798.960938,66454.101562,66653.703125,31414428647,0.026024,1396.218750,0.000000,2366.765625,1979.616908,747.104074,245.475167,37.739831,12.400135,50.537920


### Average Directional Index (ADX)

$$ADX_{t} = \frac{1}{n} \sum_{i=t-n+1}^{t} DX_{i}$$

In [576]:
df['ADX'] = df['DX'].rolling(window=ADX_LOOKBACK).mean()

df.head(ADX_LOOKBACK*3).tail(ADX_LOOKBACK)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR,SMA+DM,SMA-DM,+DI,-DI,DX,ADX
28,2024-10-29,72720.492188,73577.210938,69729.914062,69910.046875,58541874402,0.040235,3364.945312,0.000000,3847.296875,1932.910156,706.975446,282.952009,36.575701,14.638653,42.833789,39.286018
29,2024-10-30,72339.539062,72905.296875,71411.734375,72715.367188,40646637831,-0.005239,0.000000,0.000000,1493.562500,1924.124442,671.717634,282.952009,34.910301,14.705494,40.722529,38.866612
30,2024-10-31,70215.187500,72662.312500,69590.500000,72335.046875,40627912076,-0.029366,0.000000,1821.234375,3071.812500,2053.195312,671.717634,405.087612,32.715720,19.729619,24.761211,37.302077
31,2024-11-01,69482.468750,71559.015625,68779.703125,70216.898438,49989795365,-0.010435,0.000000,810.796875,2779.312500,2123.722656,596.179129,463.001674,28.072363,21.801419,12.573628,34.850771
32,2024-11-02,69289.273438,69867.351562,69033.718750,69486.023438,18184612091,-0.002780,0.000000,0.000000,833.632812,2137.313058,596.179129,463.001674,27.893861,21.662792,12.573628,32.399465
33,2024-11-03,68741.117188,69361.656250,67482.523438,69296.382812,34868307655,-0.007911,0.000000,1551.195312,1879.132812,2182.016183,546.821987,573.801339,25.060400,26.296842,2.407531,29.194875
34,2024-11-04,67811.507812,69433.179688,66803.648438,68742.132812,41184819348,-0.013523,0.000000,678.875000,2629.531250,2181.776786,546.821987,531.159040,25.063150,24.345251,1.452989,27.268935
35,2024-11-05,69359.562500,70522.789062,67458.867188,67811.171875,46046889204,0.022829,1089.609375,0.000000,3063.921875,2313.470424,624.651228,513.410156,27.000614,22.192207,9.774611,25.897910
36,2024-11-06,75639.078125,76460.156250,69322.031250,69358.500000,118592653963,0.090536,5937.367188,0.000000,7138.125000,2665.143136,1048.748884,413.886440,39.350565,15.529614,43.405382,26.839329
37,2024-11-07,75904.859375,76943.117188,74480.421875,75637.085938,63467654989,0.003514,482.960938,0.000000,2462.695312,2671.995257,983.516183,413.886440,36.808306,15.489789,40.763466,26.141153


### Market Status

In [577]:
df['Market_State'] = np.where(df['ADX'] > THETA_ADX, 'trending', 'sideways')

df.head(100).tail(30)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR,SMA+DM,SMA-DM,+DI,-DI,DX,ADX,Market_State
70,2024-12-10,96675.429688,98270.156250,94321.257812,97441.234375,104823780634,-0.007772,0.000000,34.656250,3948.898438,4117.933036,945.789621,470.777344,22.967581,11.432370,33.532638,21.659452,sideways
71,2024-12-11,101173.031250,101913.359375,95747.226562,96656.062500,85391409936,0.046523,3643.203125,0.000000,6166.132812,4159.619978,1036.773438,470.777344,24.924715,11.317797,37.544081,23.229508,sideways
72,2024-12-12,100043.000000,102524.914062,99339.953125,101167.804688,72073983533,-0.011169,611.554688,0.000000,3184.960938,4246.199219,1080.455915,470.777344,25.445248,11.087029,39.302830,24.925190,sideways
73,2024-12-13,101459.257812,101888.804688,99233.281250,100046.648438,56894751583,0.014156,0.000000,106.671875,2655.523438,4201.215960,934.529576,478.396763,22.244264,11.387102,32.282844,25.547441,trending
74,2024-12-14,101372.968750,102618.882812,100634.054688,101451.437500,40422968793,-0.000850,730.078125,0.000000,1984.828125,4246.194754,986.678013,478.396763,23.236758,11.266482,34.693195,26.341860,trending
75,2024-12-15,104298.695312,105047.539062,101227.031250,101373.531250,51145914137,0.028861,2428.656250,0.000000,3820.507812,4367.806920,1132.383371,478.396763,25.925674,10.952791,40.600613,26.836639,trending
76,2024-12-16,106029.718750,107780.578125,103322.984375,104293.578125,91020417816,0.016597,2733.039062,0.000000,4457.593750,4424.082589,1327.600446,386.445312,30.008491,8.735038,54.908402,29.261385,trending
77,2024-12-17,106140.601562,108268.445312,105291.734375,106030.687500,68589364868,0.001046,487.867188,0.000000,2976.710938,4446.159040,1362.448103,325.494978,30.643261,7.320813,61.432944,32.955610,trending
78,2024-12-18,100041.539062,106470.609375,100041.539062,106147.296875,93865656139,-0.057462,0.000000,5250.195312,6429.070312,4580.606585,1154.582031,700.508929,25.205876,15.292929,24.477134,33.394553,trending
79,2024-12-19,97490.953125,102748.148438,95587.679688,100070.687500,97221662392,-0.025495,0.000000,4453.859375,7160.468750,4241.948103,819.357701,1018.641741,19.315599,24.013536,10.842443,32.703046,trending


# Strategies

## SMA Crossover

### SMA Short and Long

In [578]:
df['SMA-Short'] = df['Close'].rolling(window=SMA_SHORT).mean()
df['SMA-Long'] = df['Close'].rolling(window=SMA_LONG).mean()

df.head(100).tail(5)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR,SMA+DM,SMA-DM,+DI,-DI,DX,ADX,Market_State,SMA-Short,SMA-Long
95,2025-01-04,98236.226562,98734.429688,97562.976562,98106.992188,22342608078,0.001313,0.000000,0.000000,1171.453125,3220.319196,591.204799,785.948103,18.358578,24.405907,14.141008,31.745729,trending,97241.358594,97088.172656
96,2025-01-05,98314.960938,98813.304688,97291.765625,98233.906250,20525254825,0.000801,0.000000,271.210938,1521.539062,3103.423549,591.204799,646.439174,19.050084,20.829873,4.462865,30.307833,trending,96855.620703,97243.302344
97,2025-01-06,102078.085938,102482.875000,97926.148438,98314.953125,51823432705,0.038276,3669.570312,0.000000,4556.726562,3142.255580,853.316964,517.935268,27.156192,16.482913,24.458060,30.778771,trending,96652.494922,97487.947031
98,2025-01-07,96922.703125,102712.484375,96132.875000,102248.851562,58685738547,-0.050504,0.000000,1793.273438,6579.609375,3186.795759,639.898996,646.026228,20.079699,20.271968,0.476484,30.322369,trending,96496.553125,97615.548281
99,2025-01-08,95043.523438,97258.320312,92525.843750,96924.164062,63875859171,-0.019388,0.000000,3607.031250,4732.476562,3390.166853,634.564174,903.671317,18.717786,26.655659,17.494535,30.132712,trending,96374.181641,97669.542969


### SMA Signal

In [579]:
df['SMA-Signal'] = np.where(df['SMA-Short'] > df['SMA-Long'], 1, 0)
df['SMA-Crossover'] = df['SMA-Signal'].diff()
# Crossover = +1 -> Long signal
# Crossover = -1 -> Short signal

df.head(100).tail(5)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,...,SMA-DM,+DI,-DI,DX,ADX,Market_State,SMA-Short,SMA-Long,SMA-Signal,SMA-Crossover
95,2025-01-04,98236.226562,98734.429688,97562.976562,98106.992188,22342608078,0.001313,0.000000,0.000000,1171.453125,...,785.948103,18.358578,24.405907,14.141008,31.745729,trending,97241.358594,97088.172656,1,0.0
96,2025-01-05,98314.960938,98813.304688,97291.765625,98233.906250,20525254825,0.000801,0.000000,271.210938,1521.539062,...,646.439174,19.050084,20.829873,4.462865,30.307833,trending,96855.620703,97243.302344,0,-1.0
97,2025-01-06,102078.085938,102482.875000,97926.148438,98314.953125,51823432705,0.038276,3669.570312,0.000000,4556.726562,...,517.935268,27.156192,16.482913,24.458060,30.778771,trending,96652.494922,97487.947031,0,0.0
98,2025-01-07,96922.703125,102712.484375,96132.875000,102248.851562,58685738547,-0.050504,0.000000,1793.273438,6579.609375,...,646.026228,20.079699,20.271968,0.476484,30.322369,trending,96496.553125,97615.548281,0,0.0
99,2025-01-08,95043.523438,97258.320312,92525.843750,96924.164062,63875859171,-0.019388,0.000000,3607.031250,4732.476562,...,903.671317,18.717786,26.655659,17.494535,30.132712,trending,96374.181641,97669.542969,0,0.0


## RSI Mean Reversion

### Return
$$\Delta_t = C_{t} - C_{t-1}$$

In [580]:
df['Return'] = df['Close'].pct_change() * 100

df.head(100).tail(5)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,...,SMA-DM,+DI,-DI,DX,ADX,Market_State,SMA-Short,SMA-Long,SMA-Signal,SMA-Crossover
95,2025-01-04,98236.226562,98734.429688,97562.976562,98106.992188,22342608078,0.131281,0.000000,0.000000,1171.453125,...,785.948103,18.358578,24.405907,14.141008,31.745729,trending,97241.358594,97088.172656,1,0.0
96,2025-01-05,98314.960938,98813.304688,97291.765625,98233.906250,20525254825,0.080148,0.000000,271.210938,1521.539062,...,646.439174,19.050084,20.829873,4.462865,30.307833,trending,96855.620703,97243.302344,0,-1.0
97,2025-01-06,102078.085938,102482.875000,97926.148438,98314.953125,51823432705,3.827622,3669.570312,0.000000,4556.726562,...,517.935268,27.156192,16.482913,24.458060,30.778771,trending,96652.494922,97487.947031,0,0.0
98,2025-01-07,96922.703125,102712.484375,96132.875000,102248.851562,58685738547,-5.050431,0.000000,1793.273438,6579.609375,...,646.026228,20.079699,20.271968,0.476484,30.322369,trending,96496.553125,97615.548281,0,0.0
99,2025-01-08,95043.523438,97258.320312,92525.843750,96924.164062,63875859171,-1.938844,0.000000,3607.031250,4732.476562,...,903.671317,18.717786,26.655659,17.494535,30.132712,trending,96374.181641,97669.542969,0,0.0


### Gain and Loss
$$G_{t} = max(\Delta_t,0)$$
$$L_{t} = max(-\Delta_t,0)$$

In [581]:
df['Gain'] = np.maximum(df['Return'], 0) * 100
df['Loss'] = np.maximum(-df['Return'], 0) * 100

df.head(100).tail(5)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,...,-DI,DX,ADX,Market_State,SMA-Short,SMA-Long,SMA-Signal,SMA-Crossover,Gain,Loss
95,2025-01-04,98236.226562,98734.429688,97562.976562,98106.992188,22342608078,0.131281,0.000000,0.000000,1171.453125,...,24.405907,14.141008,31.745729,trending,97241.358594,97088.172656,1,0.0,13.128147,0.000000
96,2025-01-05,98314.960938,98813.304688,97291.765625,98233.906250,20525254825,0.080148,0.000000,271.210938,1521.539062,...,20.829873,4.462865,30.307833,trending,96855.620703,97243.302344,0,-1.0,8.014800,0.000000
97,2025-01-06,102078.085938,102482.875000,97926.148438,98314.953125,51823432705,3.827622,3669.570312,0.000000,4556.726562,...,16.482913,24.458060,30.778771,trending,96652.494922,97487.947031,0,0.0,382.762192,0.000000
98,2025-01-07,96922.703125,102712.484375,96132.875000,102248.851562,58685738547,-5.050431,0.000000,1793.273438,6579.609375,...,20.271968,0.476484,30.322369,trending,96496.553125,97615.548281,0,0.0,0.000000,505.043053
99,2025-01-08,95043.523438,97258.320312,92525.843750,96924.164062,63875859171,-1.938844,0.000000,3607.031250,4732.476562,...,26.655659,17.494535,30.132712,trending,96374.181641,97669.542969,0,0.0,0.000000,193.884366


### Average Gain and Loss
$$\bar{G}_{t} = \frac{1}{n} \sum_{i=t-n+1}^{t} G_i$$
$$\bar{L}_{t} = \frac{1}{n} \sum_{i=t-n+1}^{t} L_i$$

In [582]:
df['Average-Gain'] = df['Gain'].rolling(window=RSI_LOOKBACK).mean()
df['Average-Loss'] = df['Loss'].rolling(window=RSI_LOOKBACK).mean()

df.head(100).tail(5)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,...,ADX,Market_State,SMA-Short,SMA-Long,SMA-Signal,SMA-Crossover,Gain,Loss,Average-Gain,Average-Loss
95,2025-01-04,98236.226562,98734.429688,97562.976562,98106.992188,22342608078,0.131281,0.000000,0.000000,1171.453125,...,31.745729,trending,97241.358594,97088.172656,1,0.0,13.128147,0.000000,84.420061,75.116252
96,2025-01-05,98314.960938,98813.304688,97291.765625,98233.906250,20525254825,0.080148,0.000000,271.210938,1521.539062,...,30.307833,trending,96855.620703,97243.302344,0,-1.0,8.014800,0.000000,84.992547,59.542693
97,2025-01-06,102078.085938,102482.875000,97926.148438,98314.953125,51823432705,3.827622,3669.570312,0.000000,4556.726562,...,30.778771,trending,96652.494922,97487.947031,0,0.0,382.762192,0.000000,112.332704,56.398081
98,2025-01-07,96922.703125,102712.484375,96132.875000,102248.851562,58685738547,-5.050431,0.000000,1793.273438,6579.609375,...,30.322369,trending,96496.553125,97615.548281,0,0.0,0.000000,505.043053,82.234414,92.472585
99,2025-01-08,95043.523438,97258.320312,92525.843750,96924.164062,63875859171,-1.938844,0.000000,3607.031250,4732.476562,...,30.132712,trending,96374.181641,97669.542969,0,0.0,0.000000,193.884366,77.723974,106.321468


### Relative Strength (RS)

$$RS_{t} = \frac{\bar{G}_{t}}{\bar{L}_{t}}$$

In [583]:
df['RS'] = df['Average-Gain'] / df['Average-Loss']

df.head(100).tail(5)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,...,Market_State,SMA-Short,SMA-Long,SMA-Signal,SMA-Crossover,Gain,Loss,Average-Gain,Average-Loss,RS
95,2025-01-04,98236.226562,98734.429688,97562.976562,98106.992188,22342608078,0.131281,0.000000,0.000000,1171.453125,...,trending,97241.358594,97088.172656,1,0.0,13.128147,0.000000,84.420061,75.116252,1.123859
96,2025-01-05,98314.960938,98813.304688,97291.765625,98233.906250,20525254825,0.080148,0.000000,271.210938,1521.539062,...,trending,96855.620703,97243.302344,0,-1.0,8.014800,0.000000,84.992547,59.542693,1.427422
97,2025-01-06,102078.085938,102482.875000,97926.148438,98314.953125,51823432705,3.827622,3669.570312,0.000000,4556.726562,...,trending,96652.494922,97487.947031,0,0.0,382.762192,0.000000,112.332704,56.398081,1.991782
98,2025-01-07,96922.703125,102712.484375,96132.875000,102248.851562,58685738547,-5.050431,0.000000,1793.273438,6579.609375,...,trending,96496.553125,97615.548281,0,0.0,0.000000,505.043053,82.234414,92.472585,0.889284
99,2025-01-08,95043.523438,97258.320312,92525.843750,96924.164062,63875859171,-1.938844,0.000000,3607.031250,4732.476562,...,trending,96374.181641,97669.542969,0,0.0,0.000000,193.884366,77.723974,106.321468,0.731028


### Relative Strength Index (RSI)
$$RSI_{t} = 100 - \frac{100}{1+RS_{t}}$$

In [584]:
df['RSI'] = 100 - 100 / (1 + df['RS'])

df.head(100).tail(5)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,...,SMA-Short,SMA-Long,SMA-Signal,SMA-Crossover,Gain,Loss,Average-Gain,Average-Loss,RS,RSI
95,2025-01-04,98236.226562,98734.429688,97562.976562,98106.992188,22342608078,0.131281,0.000000,0.000000,1171.453125,...,97241.358594,97088.172656,1,0.0,13.128147,0.000000,84.420061,75.116252,1.123859,52.915891
96,2025-01-05,98314.960938,98813.304688,97291.765625,98233.906250,20525254825,0.080148,0.000000,271.210938,1521.539062,...,96855.620703,97243.302344,0,-1.0,8.014800,0.000000,84.992547,59.542693,1.427422,58.804031
97,2025-01-06,102078.085938,102482.875000,97926.148438,98314.953125,51823432705,3.827622,3669.570312,0.000000,4556.726562,...,96652.494922,97487.947031,0,0.0,382.762192,0.000000,112.332704,56.398081,1.991782,66.575109
98,2025-01-07,96922.703125,102712.484375,96132.875000,102248.851562,58685738547,-5.050431,0.000000,1793.273438,6579.609375,...,96496.553125,97615.548281,0,0.0,0.000000,505.043053,82.234414,92.472585,0.889284,47.069902
99,2025-01-08,95043.523438,97258.320312,92525.843750,96924.164062,63875859171,-1.938844,0.000000,3607.031250,4732.476562,...,96374.181641,97669.542969,0,0.0,0.000000,193.884366,77.723974,106.321468,0.731028,42.230861


### RSI Signal

In [585]:
conditions = [df['RSI'] > 70, df['RSI'] < 30]
choices =    [      -1      ,       +1      ]
df['RSI-Signal'] = np.select(conditions, choices, default=0)
df['RSI-Crossover-temp'] = df['RSI-Signal'].diff()
df['RSI-Crossover'] = np.where(df['RSI-Signal'] != 0, df['RSI-Crossover-temp'], 0)
df = df.drop(columns=['RSI-Crossover-temp'])

df.head(100).tail(20)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,...,SMA-Signal,SMA-Crossover,Gain,Loss,Average-Gain,Average-Loss,RS,RSI,RSI-Signal,RSI-Crossover
80,2024-12-20,97755.929688,98098.914062,92175.179688,97484.695312,105634083408,0.271796,0.000000,3412.500000,5923.734375,...,1,0.0,27.179605,0.000000,87.902756,100.227281,0.877034,46.724466,0,0.0
81,2024-12-21,97224.726562,99507.101562,96426.523438,97756.195312,51765334294,-0.543397,1408.187500,0.000000,3080.578125,...,1,0.0,0.000000,54.339734,87.883991,104.108691,0.844156,45.774657,0,0.0
82,2024-12-22,95104.937500,97360.265625,94202.187500,97218.320312,43147981314,-2.180298,0.000000,2224.335938,3158.078125,...,1,0.0,0.000000,218.029830,78.500514,119.682250,0.655908,39.610162,0,0.0
83,2024-12-23,94686.242188,96416.210938,92403.132812,95099.390625,65239002919,-0.440246,0.000000,1799.054688,4013.078125,...,1,0.0,0.000000,44.024561,78.500514,95.992137,0.817781,44.987863,0,0.0
84,2024-12-24,98676.093750,99404.062500,93448.015625,94684.343750,47114953674,4.213761,2987.851562,0.000000,5956.046875,...,1,0.0,421.376060,0.000000,108.598804,90.440400,1.200778,54.561514,0,0.0
85,2024-12-25,99299.195312,99478.750000,97593.468750,98675.914062,33700394629,0.631462,74.687500,0.000000,1885.281250,...,1,0.0,63.146152,0.000000,79.878747,90.440400,0.883220,46.899452,0,0.0
86,2024-12-26,95795.515625,99884.570312,95137.882812,99297.695312,47054980873,-3.528407,0.000000,2455.585938,4746.687500,...,1,0.0,0.000000,352.840693,79.878747,107.665240,0.741918,42.592006,0,0.0
87,2024-12-27,94164.859375,97294.843750,93310.742188,95704.976562,52419934565,-1.702226,0.000000,1827.140625,3984.101562,...,1,0.0,0.000000,170.222608,69.766968,119.823998,0.582245,36.798677,0,0.0
88,2024-12-28,95163.929688,95525.898438,94014.289062,94160.187500,24107436185,1.060980,0.000000,0.000000,1511.609375,...,1,0.0,106.097999,0.000000,77.345396,119.216513,0.648781,39.349128,0,0.0
89,2024-12-29,93530.226562,95174.875000,92881.789062,95174.054688,29635885267,-1.716725,0.000000,1132.500000,2293.085938,...,1,0.0,0.000000,171.672516,56.730387,131.478835,0.431479,30.142193,0,0.0


## Final strategy

In [586]:
df['final_strategy'] = np.where(df['Market_State'] == 'trending', df['SMA-Crossover'], df['RSI-Crossover'])

df.head(100).tail(20)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,...,SMA-Crossover,Gain,Loss,Average-Gain,Average-Loss,RS,RSI,RSI-Signal,RSI-Crossover,final_strategy
80,2024-12-20,97755.929688,98098.914062,92175.179688,97484.695312,105634083408,0.271796,0.000000,3412.500000,5923.734375,...,0.0,27.179605,0.000000,87.902756,100.227281,0.877034,46.724466,0,0.0,0.0
81,2024-12-21,97224.726562,99507.101562,96426.523438,97756.195312,51765334294,-0.543397,1408.187500,0.000000,3080.578125,...,0.0,0.000000,54.339734,87.883991,104.108691,0.844156,45.774657,0,0.0,0.0
82,2024-12-22,95104.937500,97360.265625,94202.187500,97218.320312,43147981314,-2.180298,0.000000,2224.335938,3158.078125,...,0.0,0.000000,218.029830,78.500514,119.682250,0.655908,39.610162,0,0.0,0.0
83,2024-12-23,94686.242188,96416.210938,92403.132812,95099.390625,65239002919,-0.440246,0.000000,1799.054688,4013.078125,...,0.0,0.000000,44.024561,78.500514,95.992137,0.817781,44.987863,0,0.0,0.0
84,2024-12-24,98676.093750,99404.062500,93448.015625,94684.343750,47114953674,4.213761,2987.851562,0.000000,5956.046875,...,0.0,421.376060,0.000000,108.598804,90.440400,1.200778,54.561514,0,0.0,0.0
85,2024-12-25,99299.195312,99478.750000,97593.468750,98675.914062,33700394629,0.631462,74.687500,0.000000,1885.281250,...,0.0,63.146152,0.000000,79.878747,90.440400,0.883220,46.899452,0,0.0,0.0
86,2024-12-26,95795.515625,99884.570312,95137.882812,99297.695312,47054980873,-3.528407,0.000000,2455.585938,4746.687500,...,0.0,0.000000,352.840693,79.878747,107.665240,0.741918,42.592006,0,0.0,0.0
87,2024-12-27,94164.859375,97294.843750,93310.742188,95704.976562,52419934565,-1.702226,0.000000,1827.140625,3984.101562,...,0.0,0.000000,170.222608,69.766968,119.823998,0.582245,36.798677,0,0.0,0.0
88,2024-12-28,95163.929688,95525.898438,94014.289062,94160.187500,24107436185,1.060980,0.000000,0.000000,1511.609375,...,0.0,106.097999,0.000000,77.345396,119.216513,0.648781,39.349128,0,0.0,0.0
89,2024-12-29,93530.226562,95174.875000,92881.789062,95174.054688,29635885267,-1.716725,0.000000,1132.500000,2293.085938,...,0.0,0.000000,171.672516,56.730387,131.478835,0.431479,30.142193,0,0.0,0.0
